**NOTEBOOK:** 01_carga_y_limpieza.ipynb

**OBJETIVO:** Cargar, explorar y limpiar ENAHO 2025

PASO 1 : CARGAR LIBRERIAS 

In [10]:
import gdown as gd
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.ensemble import IsolationForest
import warnings
warnings.filterwarnings('ignore')

print("\nSe han importando las librerias correctamente. (Recuerda estar en el entorno virtual ven(3.11.0))")



Se han importando las librerias correctamente. (Recuerda estar en el entorno virtual ven(3.11.0))


Opciones de visualizacion

In [11]:
# Configurar pandas para ver mejor los datos
pd.set_option('display.max_columns', None)  # Mostrar todas las columnas
pd.set_option('display.max_rows', 100)      # Máximo 100 filas en pantalla
pd.set_option('display.width', 100)         # Ancho de pantalla

print("\nOpciones de visualización configuradas")


Opciones de visualización configuradas


PASO 2 : CARGAR DATOS

Descargar datos en local

In [12]:
FILE_ID = "16IzaVtYm6K8IJNsPjk1Aev8z0i7uIlEF"

# Crear la URL de la carpeta
folder_url = f"https://drive.google.com/drive/folders/{FILE_ID}"

# Descargar toda la carpeta
gd.download_folder(url=folder_url, quiet=False, use_cookies=False)

print("Archivos descargados satisfactoriamente!")

Retrieving folder contents


Processing file 1wA0ipdtmXvhUr5pI52lVuAc_uYRilR8V modulo_01_vivienda.csv
Processing file 1UOjZoWy8Dfc_pHI2Tdztk1DIaZXABmaE modulo_02_miembros.csv
Processing file 1SSkRbob42ZS16rDJetYCfpCkC4T6fqFY modulo_03_educacion.csv
Processing file 1snInGhsAQpMtt-yJY9aHJ-nCvY75sVRV modulo_05_empleo.csv
Processing file 1oc6-5OMJQzPUiATV0ZBJomOKc6tlX4ck modulo_11_servicios.csv
Processing file 1Cpypgeyw_XWgzXtbgrJZ5J0awXrg8NoJ modulo_16_equipamiento.csv
Processing file 1QOt6xL_c_JdhExOvkfCGXOOsSQHs9UHB modulo_34_sumarias.csv
Processing file 1JNJY-5JRd-EQNA6BEHocruOo5fOkCkQg modulo_37_programas.csv


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1wA0ipdtmXvhUr5pI52lVuAc_uYRilR8V
To: c:\Users\diana\Documents\AI-TF\analisis\notebooks\enaho2025\modulo_01_vivienda.csv
100%|██████████| 36.2M/36.2M [00:58<00:00, 617kB/s] 
Downloading...
From: https://drive.google.com/uc?id=1UOjZoWy8Dfc_pHI2Tdztk1DIaZXABmaE
To: c:\Users\diana\Documents\AI-TF\analisis\notebooks\enaho2025\modulo_02_miembros.csv
100%|██████████| 15.7M/15.7M [00:01<00:00, 8.13MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1SSkRbob42ZS16rDJetYCfpCkC4T6fqFY
From (redirected): https://drive.google.com/uc?id=1SSkRbob42ZS16rDJetYCfpCkC4T6fqFY&confirm=t&uuid=2c286d2c-2a18-45c5-aba8-bcd7ec30c90a
To: c:\Users\diana\Documents\AI-TF\analisis\notebooks\enaho2025\modulo_03_educacion.csv
100%|██████████| 111M/111M [00:12<00:00, 8.68MB/s] 
Downloading...
From (original): https://drive.google.com/uc?id=1snInGh

Archivos descargados satisfactoriamente!



Download completed


Cargar los 8 modulos

In [ ]:
import os
import pandas as pd

carpeta_datos = "enaho2025"

modulos_info = {
    'mod_01': 'modulo_01_vivienda.csv',
    'mod_02': 'modulo_02_miembros.csv',
    'mod_03': 'modulo_03_educacion.csv',
    'mod_05': 'modulo_05_empleo.csv',
    'mod_11': 'modulo_11_servicios.csv',
    'mod_16': 'modulo_16_equipamiento.csv',
    'mod_34': 'modulo_34_sumarias.csv',
    'mod_37': 'modulo_37_programas.csv'
}

print("Cargando modulos...")
print("")

modulos = {}

for nombre_corto, nombre_archivo in modulos_info.items():
    ruta_archivo = os.path.join(carpeta_datos, nombre_archivo)
    
    if not os.path.exists(ruta_archivo):
        print(f"{nombre_corto}: NO ENCONTRADO - {ruta_archivo}")
        continue
    
    # Probar diferentes delimitadores
    delimitadores = [',', ';', '|', '\t']
    cargo_exitoso = False
    
    for delim in delimitadores:
        try:
            # Leer todo el archivo con este delimitador
            df_temp = pd.read_csv(
                ruta_archivo, 
                encoding='latin-1', 
                delimiter=delim,
                on_bad_lines='skip'
            )
            
            # Si tiene mas de 1 columna y las filas son razonables, es el correcto
            if df_temp.shape[1] > 1 and df_temp.shape[0] > 100:
                modulos[nombre_corto] = df_temp
                n_filas = len(modulos[nombre_corto])
                n_cols = modulos[nombre_corto].shape[1]
                print(f"{nombre_corto}: {n_filas:>8,} filas x {n_cols:>3} columnas [delim: '{delim}']")
                cargo_exitoso = True
                break
        except:
            continue
    
    if not cargo_exitoso:
        # Intentar con deteccion automatica
        try:
            df_temp = pd.read_csv(
                ruta_archivo, 
                encoding='latin-1', 
                sep=None, 
                engine='python',
                on_bad_lines='skip'
            )
            if df_temp.shape[1] > 1:
                modulos[nombre_corto] = df_temp
                n_filas = len(modulos[nombre_corto])
                n_cols = modulos[nombre_corto].shape[1]
                print(f"{nombre_corto}: {n_filas:>8,} filas x {n_cols:>3} columnas [delim: auto]")
                cargo_exitoso = True
        except:
            pass
    
    if not cargo_exitoso:
        print(f"{nombre_corto}: ERROR - No se pudo leer correctamente")

print("")
print(f"Total modulos cargados: {len(modulos)}/{len(modulos_info)}")


Cargando 8 módulos...



NameError: name 'enaho_2025' is not defined